# Primero recopilamos


In [1]:
from pathlib import Path
import pandas as pd


In [5]:

carpeta = Path(".")

archivo_catalogo = carpeta / "formulario_limpio_1.xlsx"
sheet_catalogo = "Catálogo VRAI"

archivos_output = list(carpeta.glob("output_*.xlsx")) + list(carpeta.glob("output_single_*.xlsx"))
archivos_output = [f for f in archivos_output if f.name != archivo_catalogo.name]

# Catálogo VRAI: mapeo por NRC
df_catalogo = pd.read_excel(archivo_catalogo, sheet_name=sheet_catalogo, dtype=str)

columnas_catalogo = [
    "NRC",
    "Sigla",
    "Horario Cátedra/Clase",
    "Horario Ayudantía",
    "Horario Laboratorio",
    "Horario Taller",
    "Horario Terreno",
]

mapa_catalogo = (
    df_catalogo[columnas_catalogo]
    .dropna(subset=["NRC"])
    .assign(NRC=lambda x: x["NRC"].astype(str).str.strip())
    .drop_duplicates(subset=["NRC"], keep="first")
)

# Archivos output: filas a recopilar
columnas_output = [
    "RUT UC",
    "Nombre",
    "Email",
    "NRC",
    "Unidad Académica",
]

dfs = []

for archivo in archivos_output:
    df = pd.read_excel(archivo, dtype=str, sheet_name="Con éxito")

    df = df[columnas_output].copy()
    df["NRC"] = df["NRC"].astype(str).str.strip()

    dfs.append(df)

df_final = pd.concat(dfs, ignore_index=True)

# Cruce por NRC
df_final = df_final.merge(mapa_catalogo, on="NRC", how="left")

# Orden final
df_final = df_final[
    [
        "RUT UC",
        "Nombre",
        "Email",
        "NRC",
        "Unidad Académica",
        "Sigla",
        "Horario Cátedra/Clase",
        "Horario Ayudantía",
        "Horario Laboratorio",
        "Horario Taller",
        "Horario Terreno",
    ]
]

In [6]:
df_final

,RUT UC,Nombre,Email,NRC,Unidad Académica,Sigla,Horario Cátedra/Clase,Horario Ayudantía,Horario Laboratorio,Horario Taller,Horario Terreno
0,118606K,Elaine Victoria Garcia,elaine.garcia@tufts.edu,10169,Agronomia,AGC220,CLAS = M:1220 a 1720;,NaN,NaN,NaN,NaN
1,1185799,Mathilde Marie Blandine Lagarde,mathilde.lagrd@gmail.com,10169,Agronomia,AGC220,CLAS = M:1220 a 1720;,NaN,NaN,NaN,NaN
2,118539K,Vadim MAURER-DECOUT,vadim.maurerdecout@gmail.com,10169,Agronomia,AGC220,CLAS = M:1220 a 1720;,NaN,NaN,NaN,NaN
3,1186299,Ava Louise Giere,a.giere@wustl.edu,10218,Letras,LET1005,CLAS = L:1100 a 1210; W:1100 a 1210;,NaN,NaN,TAL = V:0940 a 1050;,NaN
4,1184148,Clara Fornés Roig,fornesroigc@gmail.com,10261,Estetica,EST210A,CLAS = M:1450 a 1720;,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
595,1184598,Julian Staeva-Vieira,julian.staeva_vieira@tufts.edu,29824,Agronomia,AGC209,CLAS = L:1100 a 1330;,NaN,NaN,NaN,NaN
596,1186892,Joana Haristoy,joana.haristoy@opendeusto.es,30100,Escuela de Medicina,MED855,CLAS = J:1100 a 1330;,NaN,NaN,NaN,NaN
597,1184520,Leslie Santiago Andres,lc23-0707@lclark.edu,30374,Ciencia Politica,ICP0720,CLAS = L:1220 a 1330; W:1220 a 1330;,NaN,NaN,NaN,NaN
598,1185748,Ximena Herrera Monreal,A01705214@itesm.mx,33476,Derecho,DNO158,CLAS = V:0820 a 1050;,NaN,NaN,NaN,NaN


In [7]:

# Guardar resultado
df_final.to_excel("recopilado_outputs.xlsx", index=False)

# Ahora chequeamos

In [19]:
modulos = {
    1: "08:20",
    2: "09:40",
    3: "11:00",
    4: "12:20",
    5: "13:30",
    6: "14:50",
    7: "17:30",
    8: "18:50",
    9: "20:10"
}

In [30]:
archivo_eval = "Material Docente 24-02-2026_FACEA.xlsx"

df_eval = pd.read_excel(
    archivo_eval,
    skiprows=11,
    header=[0, 1]
)


In [26]:
df_eval

Unnamed: 0_level_0 Unnamed: 1_level_0 Unnamed: 2_level_0  \
    Unnamed: 0_level_1              Sigla               Sec.   
0                  NaN            EAA1110                  1   
1                  NaN            EAA1110                  2   
2                  NaN            EAA1110                  3   
3                  NaN            EAA1110                  4   
4                  NaN            EAA1110                  5   
..                 ...                ...                ...   
382                NaN            EAG190A                  1   
383                NaN            EAG165E                  1   
384                NaN            EAG160E                  1   
385                NaN                NaN                NaN   
386                NaN                NaN                NaN   

    Unnamed: 3_level_0                                 Unnamed: 4_level_0  \
                  Créd                                   Nombre del curso   
0                   10                      COMPORTAMIENTO ORGANIZACIONAL   
1                   10                      COMPORTAMIENTO ORGANIZACIONAL   
2                   10                      COMPORTAMIENTO ORGANIZACIONAL   
3                   10                      COMPORTAMIENTO ORGANIZACIONAL   
4                   10                      COMPORTAMIENTO ORGANIZACIONAL   
..                 ...                                                ...   
382                 10  EXPLORANDO EL MUNDO DEL TRABAJO PARA LA TOMA D...   
383                 10  POLÍTICAS PÚBLICAS: DERRIBANDO MITOS DE LA POB...   
384                 10                          ¿CÓMO TOMAMOS DECISIONES?   
385                NaN                                                NaN   
386                NaN                                                NaN   

          Unnamed: 5_level_0                    Unnamed: 6_level_0  Clases  \
                        Prof                            Requisitos Horario   
0                   G. MUÑOZ                              ADMISIÓN   M-J:2   
1              S. VALENZUELA                              ADMISIÓN   M-J:4   
2              M.E. VIGNEAUX                              ADMISIÓN   M-J:4   
3                  E. KAUSEL                              ADMISIÓN   L-W:3   
4              S. VALENZUELA                              ADMISIÓN   M-J:5   
..                       ...                                   ...     ...   
382                  K. LUND                     CIENCIAS SOCIALES   M-J:5   
383  G. UGARTE / F. BASTIDAS                     CIENCIAS SOCIALES   L-W:2   
384                 C. MUÑOZ                     CIENCIAS SOCIALES   M-J:6   
385                      NaN                                   NaN     NaN   
386                      NaN  CURSOS DICTADOS POR OTRAS FACULTADES     NaN   

    Ayudantía                        Pruebas               \
      Horario Horario.1           Día/Módulo Día/Módulo.1   
0         V:6       NaN  29/abr:7 y 29/may:7          NaN   
1         V:6       NaN  29/abr:7 y 29/may:7          NaN   
2         V:6       NaN  29/abr:7 y 29/may:7          NaN   
3         V:6       NaN  29/abr:7 y 29/may:7          NaN   
4         V:6       NaN  29/abr:7 y 29/may:7          NaN   
..        ...       ...                  ...          ...   
382       V:5       NaN                   PF          NaN   
383       V:2       NaN             24/abr:2          NaN   
384         -       NaN   9/abr:6 y 14/may:6          NaN   
385       NaN       NaN                  NaN          NaN   
386       NaN       NaN                  NaN          NaN   

                  Examen                        Prueba Recuperativa            
                     Día           Hora Hora.1                  Día      Hora  
0    2024-07-03 00:00:00   8:20 - 10:20    NaN  2024-07-03 00:00:00  17:30:00  
1    2024-07-03 00:00:00   8:20 - 10:20    NaN  2024-07-03 00:00:00  17:30:00  
2    2024-07-03 00:00:00   8:20 - 10:20 

In [31]:

df_eval = df_eval[[("Unnamed: 1_level_0", "Sigla"),
                   ("Pruebas", "Día/Módulo"),
                   ("Examen", "Día"),
                   ("Examen", "Hora")]]

df_eval.columns = ["Sigla", "Pruebas", "Examen Dia", "Examen Hora"]
df_eval = df_eval.dropna(subset=["Sigla"]).reset_index(drop=True)
df_eval = df_eval.drop_duplicates(subset="Sigla").reset_index(drop=True)

In [32]:
df_eval

,Sigla,Pruebas,Examen Dia,Examen Hora
0,EAA1110,29/abr:7 y 29/may:7,2024-07-03 00:00:00,8:20 - 10:20
1,EAA1210,13/abr:7 y 26/may:7,2025-07-09 00:00:00,8:20 - 10:20
2,EAA1220,13/abr:7 y 26/may:7,2025-07-10 00:00:00,8:20 - 10:20
3,EAA2230,13/abr:7 y 26/may:7,2024-06-30 00:00:00,13:30 - 15:30
4,EAA220B,14/abr:7 y 1/jun:7,2024-07-09 00:00:00,13:30 - 15:30
...,...,...,...,...
169,CURSOS DE FORMACION GENERAL DICTADOS POR LA FA...,NaN,NaN,NaN
170,EAG170A,17/abr:2,2025-07-01 00:00:00,13:30 - 15:30
171,EAG190A,PF,PF,PF
172,EAG165E,24/abr:2,2025-07-01 00:00:00,13:30 - 15:30


In [33]:
siglas_relevantes = set(df_final["Sigla"].unique())

df_eval = df_eval[df_eval["Sigla"].isin(siglas_relevantes)].reset_index(drop=True)

In [34]:
df_eval

,Sigla,Pruebas,Examen Dia,Examen Hora
0,EAA1220,13/abr:7 y 26/may:7,2025-07-10 00:00:00,8:20 - 10:20
1,EAA2220,10/abr:7 y 15/may:7,2024-07-02 00:00:00,8:20 - 10:20
2,EAA2310,29/abr:7 y 29/may:7,2024-07-04 00:00:00,8:20 - 10:20
3,EAA2410,13/abr:7 y 26/may:7,2024-07-02 00:00:00,8:20 - 10:20
4,EAA2110,30/abr:7 y 11/jun:7,2024-07-03 00:00:00,8:20 - 10:20
5,EAA2420,23/abr:7 y 10/jun:7,2024-07-06 00:00:00,13:30 - 15:30
6,EAA1510,27/abr:7 y 5/jun:7,2024-07-08 00:00:00,13:30 - 15:30
7,EAA2081,PF,PF,PF
8,EAA2091,8/abr:7 y 2/jun:7,2025-06-30 00:00:00,8:20 - 10:20
9,EAA209L,PF,PF,PF
